In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_3.py ──
"""
Shared infrastructure for MLFP03 Exercise 3 — The Classical ML Zoo.

Contains: e-commerce churn data loading, preprocessing, CV strategy,
2D PCA projection for decision boundary plots, model comparison helpers,
and a shared ModelVisualizer-backed plot utility.

Technique-specific code (model fitting, parameter sweeps, from-scratch
Gini, OOB convergence, decision guide) does NOT belong here — it lives
in the per-technique files under modules/mlfp03/solutions/ex_3/.
"""

import time
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

from kailash_ml import ModelVisualizer, PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()
np.random.seed(42)

# Output directory for comparison artifacts (HTML plots, tables)
OUTPUT_DIR = Path("outputs") / "ex3_model_zoo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# E-commerce churn dataset — Singapore APAC retail churn scenario
DATASET_MODULE = "mlfp03"
DATASET_FILE = "ecommerce_customers.parquet"
TARGET_COL = "churned"

# SVM with RBF kernel is O(n²) — cap the training set so every technique
# in the zoo fits in a few seconds on a laptop.
SUBSAMPLE_N = 5000
RANDOM_SEED = 42

# Drop columns that are text/ID or leak the target
DROP_COLS = ["customer_id", "review_text", "product_categories"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING + PREPROCESSING
# ════════════════════════════════════════════════════════════════════════


def load_ecommerce_churn() -> pl.DataFrame:
    """Load the Singapore e-commerce churn dataset (polars DataFrame).

    Drops text/ID columns and subsamples for SVM tractability.
    """
    loader = MLFPDataLoader()
    df = loader.load(DATASET_MODULE, DATASET_FILE)
    df = df.sample(n=min(SUBSAMPLE_N, df.height), seed=RANDOM_SEED)
    keep = [c for c in df.columns if c not in DROP_COLS]
    return df.select(keep)


def build_train_test_split() -> dict[str, Any]:
    """Return a fully prepared dict: X_train, X_test, y_train, y_test, feature_names, cv.

    Uses kailash_ml PreprocessingPipeline with z-score normalisation and
    ordinal categorical encoding. Every technique file calls this so all
    models share identical folds and identical preprocessing.
    """
    df = load_ecommerce_churn()

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=df,
        target=TARGET_COL,
        train_size=0.8,
        seed=RANDOM_SEED,
        normalize=True,
        normalize_method="zscore",
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != TARGET_COL]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=TARGET_COL,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=TARGET_COL,
    )
    feature_names = col_info["feature_columns"]

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "feature_names": feature_names,
        "cv": cv,
        "churn_rate": float(np.mean(y_train)),
    }


# ════════════════════════════════════════════════════════════════════════
# 2D PCA PROJECTION — shared so every technique plots on the same axes
# ════════════════════════════════════════════════════════════════════════


def project_2d(X_train: np.ndarray, X_test: np.ndarray) -> dict[str, Any]:
    """Fit PCA(2) on X_train and project both train and test.

    Returns {X_train_2d, X_test_2d, explained_variance, pca}.
    """
    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    X_train_2d = pca.fit_transform(X_train)
    X_test_2d = pca.transform(X_test)
    return {
        "X_train_2d": X_train_2d,
        "X_test_2d": X_test_2d,
        "explained_variance": pca.explained_variance_ratio_,
        "pca": pca,
    }


# ════════════════════════════════════════════════════════════════════════
# CROSS-VALIDATION HELPER — keep every parameter sweep one line
# ════════════════════════════════════════════════════════════════════════


def cv_accuracy_f1(
    estimator: Any,
    X: np.ndarray,
    y: np.ndarray,
    cv: Any,
) -> tuple[float, float]:
    """Return (mean_accuracy, mean_f1) for a 5-fold CV."""
    acc = cross_val_score(estimator, X, y, cv=cv, scoring="accuracy").mean()
    f1 = cross_val_score(estimator, X, y, cv=cv, scoring="f1").mean()
    return float(acc), float(f1)


# ════════════════════════════════════════════════════════════════════════
# EVALUATION — train on full set, measure timing, return pred/prob/metrics
# ════════════════════════════════════════════════════════════════════════


def fit_and_evaluate(
    estimator: Any,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    name: str,
) -> dict[str, Any]:
    """Fit, predict, score, and time a single model.

    Returns a dict with keys: name, model, pred, prob, train_time,
    accuracy, f1, auc_roc.
    """
    t0 = time.perf_counter()
    estimator.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    pred = estimator.predict(X_test)
    if hasattr(estimator, "predict_proba"):
        prob = estimator.predict_proba(X_test)[:, 1]
    else:
        # Decision-function fallback (never used by the zoo but keeps contract)
        prob = estimator.decision_function(X_test)

    return {
        "name": name,
        "model": estimator,
        "pred": pred,
        "prob": prob,
        "train_time": float(train_time),
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred)),
        "auc_roc": float(roc_auc_score(y_test, prob)),
    }


def print_classification_report(y_test: np.ndarray, pred: np.ndarray) -> None:
    """Print sklearn classification report with churn-friendly target names."""
    print(
        classification_report(
            y_test,
            pred,
            target_names=["Retained", "Churned"],
        )
    )


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION — Plotly via kailash_ml.ModelVisualizer
# ════════════════════════════════════════════════════════════════════════


def get_visualizer() -> ModelVisualizer:
    """Return a ModelVisualizer instance (polars-native plots)."""
    return ModelVisualizer()


def save_metric_comparison(
    metric_dict: dict[str, dict[str, float]], fname: str
) -> Path:
    """Save a metric_comparison plot to OUTPUT_DIR/fname and return the path."""
    viz = get_visualizer()
    fig = viz.metric_comparison(metric_dict)
    fig.update_layout(title="Classical ML Zoo — Performance Comparison")
    out = OUTPUT_DIR / fname
    fig.write_html(str(out))
    return out


# ════════════════════════════════════════════════════════════════════════
# DECISION BOUNDARY MESH — shared helper so every technique file uses
# the same axes, grid, and figure style.
# ════════════════════════════════════════════════════════════════════════


def decision_boundary_mesh(
    X_2d: np.ndarray,
    step: float = 0.1,
    pad: float = 0.5,
) -> tuple[np.ndarray, np.ndarray]:
    """Return (xx, yy) meshgrid covering the 2D PCA projection."""
    x_min, x_max = X_2d[:, 0].min() - pad, X_2d[:, 0].max() + pad
    y_min, y_max = X_2d[:, 1].min() - pad, X_2d[:, 1].max() + pad
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, step),
        np.arange(y_min, y_max, step),
    )
    return xx, yy


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE E-COMMERCE CHURN — business-impact constants
# ════════════════════════════════════════════════════════════════════════
# Public industry figures used for the "Apply" phases. Sources in reading
# notes (SGX retail analyst reports, Shopee/Lazada 2024 ops reviews).

AVG_CUSTOMER_LIFETIME_VALUE_SGD = 420.0  # avg 12-month CLV per retained SG customer
RETENTION_OFFER_COST_SGD = 18.0  # targeted promo cost per flagged customer
MONTHLY_ACTIVE_CUSTOMERS = 250_000  # typical mid-market SG e-commerce platform
ANNUAL_CHURN_BASELINE = 0.22  # industry baseline annual churn


def churn_saved_dollars(true_positives: int) -> float:
    """Dollar value of correctly identified churners (retention offer accepted).

    Assumes a 40% offer-acceptance rate and the retained lifetime value
    net of offer cost. Public industry benchmarks — not proprietary data.
    """
    accept_rate = 0.40
    net_value_per_save = AVG_CUSTOMER_LIFETIME_VALUE_SGD - RETENTION_OFFER_COST_SGD
    return round(true_positives * accept_rate * net_value_per_save, 2)




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 3.5: Random Forests (bagging + OOB)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Bagging: bootstrap sampling + feature subsampling
#   - OOB (out-of-bag) estimation as a free cross-validation proxy
#   - The (1 - 1/n)^n -> 1/e result behind OOB coverage
#   - Read Random Forest feature importances and compare to a single tree
#   - Visualise OOB convergence as the forest grows
#   - Apply a robust, drop-in churn model to Singapore e-commerce scale
#
# PREREQUISITES: 04_decision_tree.py
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — bagging, OOB, feature importance
#   2. Build — RF with 200 trees, sqrt feature subsampling
#   3. Train — inspect OOB score and compare to CV
#   4. Visualise — OOB convergence curve + importance bar chart
#   5. Apply — Singapore e-commerce marketplace churn at 250K MAU
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import math

import numpy as np
from dotenv import load_dotenv
from sklearn.ensemble import RandomForestClassifier


load_dotenv()



## THEORY — Bagging and OOB Estimation


In [ ]:
# A Random Forest is a bag of decision trees where:
#   1. Each tree is trained on a bootstrap sample of the training data
#      (n rows sampled with replacement).
#   2. At every split, the tree considers only a random subset of
#      features (sqrt(n_features) by default for classification).
#
# These two sources of randomness DE-CORRELATE the trees: each one
# makes different mistakes, so averaging their votes cancels noise and
# leaves the signal.
#
# OOB estimation: for a bootstrap sample of size n, each row has a
#     P(NOT in sample) = (1 - 1/n)^n
# probability of being absent. As n -> infinity this tends to 1/e
# ≈ 0.368. So roughly 36.8% of rows are OUT of any given tree's
# training set. We evaluate each row using only the trees that did NOT
# see it — a free cross-validation proxy baked into training itself.



## TASK 2 — BUILD: fit a 200-tree Random Forest


In [ ]:
print("\n" + "=" * 70)
print("  MLFP03 Exercise 3.5 — Random Forests")
print("=" * 70)

data = build_train_test_split()
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
cv = data["cv"]
feature_names = data["feature_names"]

print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")

# Verify the 1/e OOB fraction analytically
n = X_train.shape[0]
oob_fraction_formula = (1 - 1 / n) ** n
print(
    f"\nOOB fraction formula (1 - 1/n)^n for n={n}: "
    f"{oob_fraction_formula:.4f}  (asymptote 1/e = {1/math.e:.4f})"
)



## TASK 3 — TRAIN: inspect OOB score and CV consistency


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    oob_score=True,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf_result = fit_and_evaluate(
    rf,
    X_train,
    y_train,
    X_test,
    y_test,
    name="RandomForest (200 trees)",
)
rf_model = rf_result["model"]

print(
    f"\n{rf_result['name']}: trained in {rf_result['train_time']:.2f}s | "
    f"accuracy={rf_result['accuracy']:.4f} | "
    f"F1={rf_result['f1']:.4f} | AUC={rf_result['auc_roc']:.4f}"
)
print(f"OOB score: {rf_model.oob_score_:.4f}")
print_classification_report(y_test, rf_result["pred"])

rf_cv_acc, rf_cv_f1 = cv_accuracy_f1(
    RandomForestClassifier(
        n_estimators=100,
        max_features="sqrt",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
    X_train,
    y_train,
    cv,
)
print(
    f"5-fold CV — accuracy: {rf_cv_acc:.4f} | F1: {rf_cv_f1:.4f} "
    f"(OOB should be within 10pp)"
)



## TASK 4 — VISUALISE: OOB convergence + importance


In [ ]:
print("\n--- OOB convergence vs number of trees ---")
n_trees_grid = [10, 25, 50, 75, 100, 150, 200]
oob_scores: list[float] = []
print(f"{'trees':>8} {'OOB score':>12}")
print("-" * 24)
for n_trees in n_trees_grid:
    rf_tmp = RandomForestClassifier(
        n_estimators=n_trees,
        max_features="sqrt",
        oob_score=True,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    rf_tmp.fit(X_train, y_train)
    oob_scores.append(float(rf_tmp.oob_score_))
    print(f"{n_trees:>8} {rf_tmp.oob_score_:>12.4f}")

importances = sorted(
    zip(feature_names, rf_model.feature_importances_),
    key=lambda pair: pair[1],
    reverse=True,
)
print("\n--- Top feature importances ---")
print(f"{'feature':<30} {'importance':>12}")
print("-" * 44)
for name, imp in importances[:10]:
    bar = "#" * int(imp * 50)
    print(f"{name:<30} {imp:>12.4f}  {bar}")

viz = get_visualizer()
fig_oob = viz.training_history(
    {"OOB score": oob_scores},
    x_label="trees (index into n_trees_grid)",
)
fig_oob.update_layout(title="Random Forest — OOB score vs number of trees")
out_oob = OUTPUT_DIR / "ex3_05_rf_oob.html"
fig_oob.write_html(str(out_oob))
print(f"\nSaved: {out_oob}")

fig_imp = viz.training_history(
    {"importance": [imp for _, imp in importances[:10]]},
    x_label="feature rank (top 10)",
)
fig_imp.update_layout(title="Random Forest — top 10 feature importances")
out_imp = OUTPUT_DIR / "ex3_05_rf_importance.html"
fig_imp.write_html(str(out_imp))
print(f"Saved: {out_imp}")

# 2D decision boundary
pca_bundle = project_2d(X_train, X_test)
X_train_2d = pca_bundle["X_train_2d"]
rf_2d = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf_2d.fit(X_train_2d, y_train)
xx, yy = decision_boundary_mesh(X_train_2d)
Z = rf_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
print(
    f"\nDecision mesh shape: {Z.shape} | "
    f"PCA variance captured: {pca_bundle['explained_variance'].sum():.2%}"
)
print("RF boundaries look like 'voted' trees — axis-aligned but smoothed.")



### Checkpoint 1


In [ ]:
assert rf_result["accuracy"] > 0.5, "Random Forest must beat random"
assert rf_model.oob_score_ > 0.5, "OOB score should beat random"
assert abs(rf_model.oob_score_ - rf_cv_acc) < 0.10, "OOB and CV within 10pp"
print("\n[ok] Checkpoint 1 passed — OOB + CV consistent, RF trained and visualised\n")



## TASK 5 — APPLY: 250K MAU Singapore marketplace churn model


In [ ]:
# SCENARIO: A mid-market Singapore e-commerce platform wants a single
# production churn model to replace a stack of hand-tuned heuristics.
# Requirements:
#   - Robust across seasonal shifts (11.11, 12.12, Chinese New Year)
#   - Tolerant of mixed feature types and missing values
#   - A single well-understood hyperparameter (number of trees) the
#     retention team can scale up or down based on nightly compute
#     budget
#
# Why Random Forest fits:
#   - Bagging + feature subsampling produce calibrated, robust models
#     with minimal tuning. "A Random Forest of 200 trees on whatever
#     cleaned features we have" is the single most reliable baseline
#     in tabular ML.
#   - OOB gives a trustworthy accuracy estimate without a holdout
#     split, which saves training data for regions with thin coverage
#     (e.g. new campaign cohorts).
#   - Feature importance lists give the retention team a plain-English
#     story for every model refresh.
#
# LIMITATIONS:
#   - Memory: 200 deep trees on 250K customers is a multi-gigabyte
#     model. Move to gradient boosting (Exercise 4) for tighter limits.
#   - Black-box per-prediction: individual predictions don't have a
#     single clean decision path. For that, drop back to a single
#     decision tree (04_decision_tree.py).

true_positives = int(((rf_result["pred"] == 1) & (y_test == 1)).sum())
dollars_saved = churn_saved_dollars(true_positives)
print(f"\nBusiness impact on held-out test set ({len(y_test)} customers):")
print(f"  True positives (churners caught): {true_positives}")
print(f"  Net retention value at 40% offer acceptance: S${dollars_saved:,.2f}")
monthly_scale = dollars_saved * (250_000 / len(y_test))
print(
    f"  Extrapolated to 250K monthly active base: "
    f"S${monthly_scale:,.0f} / month retained value"
)



## REFLECTION


[x] Bagging — bootstrap + feature subsampling de-correlates trees
  [x] OOB score as a free cross-validation proxy
  [x] The (1 - 1/n)^n -> 1/e result behind OOB coverage
  [x] Held-out accuracy: {rf_result['accuracy']:.4f}, F1: {rf_result['f1']:.4f}
  [x] OOB convergence curve and feature importance plot
  [x] 250K MAU churn business case
      — S${monthly_scale:,.0f}/month retained value at scale

  Next: 06_model_zoo.py — direct head-to-head comparison across all 5.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

